# Exemplos de Keras: Pretrained, Fine-tuning e U-Net
Este notebook demonstra o uso de redes pré-treinadas, fine-tuning e uma arquitetura U-Net em Keras.

## 1. Rede pré-treinada no Keras
Utiliza um modelo pré-treinado em ImageNet como extrator de características e adiciona uma cabeça densa para classificação do seu conjunto de dados.

In [ ]:
from tensorflow.keras.applications import VGG16
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

# Carregar base pré-treinada sem o topo
base_model = VGG16(include_top=False, weights='imagenet', input_shape=(100, 100, 3))
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dropout(0.5)(x)
x = Dense(128, activation='relu')(x)
predictions = Dense(2, activation='softmax')(x)

model_pretrained = Model(inputs=base_model.input, outputs=predictions)
model_pretrained.compile(optimizer=Adam(learning_rate=1e-4),
                         loss='categorical_crossentropy',
                         metrics=['accuracy'])

model_pretrained.summary()

ModuleNotFoundError: No module named 'keras'

## 2. Fine-tuning no Keras
Depois de treinar a nova cabeça, descongela algumas camadas finais do modelo pré-treinado e treina novamente com uma taxa de aprendizado baixa.

In [ ]:
# Descongelar as últimas camadas do modelo base
base_model.trainable = True
for layer in base_model.layers[:-4]:
    layer.trainable = False

model_pretrained.compile(optimizer=Adam(learning_rate=1e-5),
                         loss='categorical_crossentropy',
                         metrics=['accuracy'])

# Exemplo de chamada de fit (ajuste o conjunto de dados conforme disponível)
# model_pretrained.fit(train_generator, epochs=10, validation_data=validation_generator)

## 3. Rede U-Net no Keras
U-Net é uma arquitetura de segmentação que combina um caminho de contração com um caminho de expansão e conexões de salto.

In [ ]:
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Conv2DTranspose, concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam

def build_unet(input_shape=(128, 128, 1)):
    inputs = Input(shape=input_shape)
    
    # Encoder
    c1 = Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
    c1 = Conv2D(32, (3, 3), activation='relu', padding='same')(c1)
    p1 = MaxPooling2D((2, 2))(c1)
    
    c2 = Conv2D(64, (3, 3), activation='relu', padding='same')(p1)
    c2 = Conv2D(64, (3, 3), activation='relu', padding='same')(c2)
    p2 = MaxPooling2D((2, 2))(c2)
    
    # Bottleneck
    c3 = Conv2D(128, (3, 3), activation='relu', padding='same')(p2)
    c3 = Conv2D(128, (3, 3), activation='relu', padding='same')(c3)
    
    # Decoder
    u4 = Conv2DTranspose(64, (2, 2), strides=(2, 2), padding='same')(c3)
    u4 = concatenate([u4, c2])
    c4 = Conv2D(64, (3, 3), activation='relu', padding='same')(u4)
    c4 = Conv2D(64, (3, 3), activation='relu', padding='same')(c4)
    
    u5 = Conv2DTranspose(32, (2, 2), strides=(2, 2), padding='same')(c4)
    u5 = concatenate([u5, c1])
    c5 = Conv2D(32, (3, 3), activation='relu', padding='same')(u5)
    c5 = Conv2D(32, (3, 3), activation='relu', padding='same')(c5)
    
    outputs = Conv2D(1, (1, 1), activation='sigmoid')(c5)
    return Model(inputs=[inputs], outputs=[outputs])

unet_model = build_unet()
unet_model.compile(optimizer=Adam(learning_rate=1e-4),
                   loss='binary_crossentropy',
                   metrics=['accuracy'])
unet_model.summary()

# Exemplo de uso para segmentação:
# unet_model.fit(x_train, y_train, epochs=20, batch_size=16, validation_split=0.1)